In [ ]:
# Instalar librerías de visión por computadora e IA
!pip install ultralytics easyocr opencv-python pandas matplotlib

In [ ]:
import os
import cv2
import pandas as pd
import easyocr
from ultralytics import YOLO
import matplotlib.pyplot as plt

# 1. Inicializar los modelos de IA
print("Cargando modelos de IA...")
# YOLOv8 nano preentrenado (detecta 80 clases, usaremos la clase 0: 'person')
yolo_model = YOLO('yolov8n.pt')
# Inicializar lector OCR en español e inglés
reader = easyocr.Reader(['es', 'en'], gpu=True)

# 2. Estructura de almacenamiento de datos (DataFrame)
datos_serial = []

def procesar_imagen_runner(ruta_imagen):
    """
    Procesa una imagen para extraer el número de bib (corredor)
    y preparar la extracción multi-objetivo.
    """
    nombre_archivo = os.path.basename(ruta_imagen)
    img = cv2.imread(ruta_imagen)
    if img is None:
        return

    h, w, _ = img.shape

    # Ejecutar detección de personas con YOLO
    resultados = yolo_model(img, verbose=False)[0]

    # Iterar sobre cada objeto detectado
    for box in resultados.boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])

        # Filtro: Solo nos interesan detecciones de personas (clase 0) con confianza > 40%
        if cls == 0 and conf > 0.40:
            # Obtener coordenadas de la caja de la persona [x_min, y_min, x_max, y_max]
            xyxy = box.xyxy[0].tolist()
            x1, y1, x2, y2 = map(int, xyxy)

            # --- HEURÍSTICA DE INGENIERÍA DE PROCESOS ---
            # El número de competidor (bib) se encuentra en el torso/pecho.
            # Recortamos el tercio superior del cuerpo detectado para optimizar el OCR.
            alto_persona = y2 - y1
            torso_y1 = y1 + int(alto_persona * 0.15) # Bajamos un 15% desde la cabeza
            torso_y2 = y1 + int(alto_persona * 0.60) # Cortamos a la altura de la cintura

            # Evitar salirnos de los límites de la imagen original
            torso_y1, torso_y2 = max(0, torso_y1), min(h, torso_y2)

            # Recortar región de interés (ROI)
            torso_crop = img[torso_y1:torso_y2, x1:x2]

            # 3. MÓDULO 1: Extracción del Número de Competidor (OCR)
            # Ejecutar OCR en el recorte del torso
            ocr_results = reader.readtext(torso_crop, allowlist='0123456789') # Solo buscar números

            bib_detectado = "No detectado"
            for (bbox, text, ocr_conf) in ocr_results:
                if len(text) >= 2 and ocr_conf > 0.30: # Filtrar ruido visual de un solo dígito
                    bib_detectado = text
                    break # Tomamos el primer número consistente

            # --- ESPACIO PARA MÓDULOS 2 Y 3 (FUTURAS FASES) ---
            club_detectado = "Pendiente de análisis" # Módulo Patrocinadores / Equipos
            wearables = "Pendiente de análisis"       # Módulo Tecnología
            emocion = "Pendiente de análisis"         # Módulo Comunidad

            # Guardar la información estructurada
            datos_serial.append({
                "Archivo": nombre_archivo,
                "Bib_Corredor": bib_detectado,
                "Club": club_detectado,
                "Wearables_Gadgets": wearables,
                "Nivel_Emocion": emocion
            })

    print(f"✓ {nombre_archivo} procesado exitosamente.")

# --- PRUEBA CON UNA SOLA IMAGEN ---
# Nota: Para probarlo en Colab, puedes arrastrar una de tus fotos a la pestaña de "Archivos"
# en el panel izquierdo y renombrarla como 'prueba.jpg'
ruta_prueba = 'prueba.jpg'

if os.path.exists(ruta_prueba):
    procesar_imagen_runner(ruta_prueba)
    # Convertir a DataFrame de Pandas para visualizar el resultado tabular
    df_resultado = pd.DataFrame(datos_serial)
    display(df_resultado)
else:
    print(f"\n[!] Por favor, sube una imagen con el nombre '{ruta_prueba}' al panel izquierdo de Colab para ejecutar la prueba inicial.")

In [ ]:
!pip install transformers torch

In [ ]:
import os
import cv2
import pandas as pd
import easyocr
import torch
from ultralytics import YOLO
from transformers import CLIPProcessor, CLIPModel
from PIL import Image

print("Cargando ecosistema de IA (YOLOv8 + EasyOCR + CLIP)...")
# 1. Inicializar Modelos
yolo_model = YOLO('yolov8n.pt')
reader = easyocr.Reader(['es', 'en'], gpu=True)

# Cargar CLIP de OpenAI para clasificación Zero-Shot de atributos y emociones
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

datos_serial = []

# Listas de prompts en inglés para que CLIP clasifique con alta precisión
prompts_wearables = ["a runner wearing sports sunglasses", "a runner wearing headphones", "a runner wearing a smartwatch", "a runner with no gadgets"]
prompts_emociones = ["a runner smiling or making a happy gesture like a heart", "a runner with a neutral focused expression", "a runner looking exhausted"]

def procesar_imagen_multi_objetivo(ruta_imagen):
    img = cv2.imread(ruta_imagen)
    if img is None: return
    h, w, _ = img.shape
    nombre_archivo = os.path.basename(ruta_imagen)

    resultados = yolo_model(img, verbose=False)[0]

    for box in resultados.boxes:
        cls = int(box.cls[0])
        conf = float(box.conf[0])

        if cls == 0 and conf > 0.45: # Filtrar personas con buena confianza
            xyxy = box.xyxy[0].tolist()
            x1, y1, x2, y2 = map(int, xyxy)

            # Recortes estratégicos
            alto = y2 - y1
            torso_y1, torso_y2 = max(0, y1 + int(alto*0.15)), min(h, y1 + int(alto*0.65))

            cuerpo_crop = img[y1:y2, x1:x2]
            torso_crop = img[torso_y1:torso_y2, x1:x2]

            if cuerpo_crop.size == 0 or torso_crop.size == 0: continue

            # --- MÓDULO 1 & 2: OCR ABIERTO (Bibs y Clubes) ---
            ocr_results = reader.readtext(torso_crop)
            bib_detectado = "No detectado"
            club_detectado = "Genérico / Independiente"

            for (bbox, text, ocr_conf) in ocr_results:
                text_clean = text.strip().upper()
                if text_clean.isdigit() and len(text_clean) >= 2 and bib_detectado == "No detectado":
                    bib_detectado = text_clean
                elif any(word in text_clean for word in ["JABALI", "JABALÍES", "CALICO", "CALICOS", "MUEVETE", "MEXICO"]):
                    club_detectado = text_clean

            # Si el OCR no detectó texto de club pero sabemos que traen jerseys específicos:
            # Convertir crop de BGR (OpenCV) a RGB (PIL) para CLIP
            cuerpo_rgb = Image.fromarray(cv2.cvtColor(cuerpo_crop, cv2.COLOR_BGR2RGB))

            # --- MÓDULO 2: WEARABLES (CLIP Zero-Shot) ---
            inputs_w = clip_processor(text=prompts_wearables, images=cuerpo_rgb, return_tensors="pt", padding=True)
            outputs_w = clip_model(**inputs_w)
            probs_w = outputs_w.logits_per_image.softmax(dim=1)
            idx_w = torch.argmax(probs_w).item()
            wearable_final = prompts_wearables[idx_w].replace("a runner wearing ", "").replace("a runner with ", "")

            # --- MÓDULO 3: EMOCIÓN Y COMUNIDAD (CLIP Zero-Shot) ---
            inputs_e = clip_processor(text=prompts_emociones, images=cuerpo_rgb, return_tensors="pt", padding=True)
            outputs_e = clip_model(**inputs_e)
            probs_e = outputs_e.logits_per_image.softmax(dim=1)
            idx_e = torch.argmax(probs_e).item()
            emocion_final = "Alta Energía / Positiva" if idx_e == 0 else ("Concentrado" if idx_e == 1 else "Agotado")

            # Filtrar ruido de fondo: si no hay número ni club ni alta confianza de persona, omitir para limpiar el CSV
            if bib_detectado == "No detectado" and conf < 0.60:
                continue

            datos_serial.append({
                "Archivo": nombre_archivo,
                "Bib_Corredor": bib_detectado,
                "Club / Jersey": club_detectado,
                "Gadgets_Detectados": wearable_final,
                "Nivel_Emocion": emocion_final
            })

    print(f"✓ {nombre_archivo} procesado con analítica multi-objetivo.")

# Ejecutar la prueba estructurada
ruta_prueba = 'prueba.jpg'
if os.path.exists(ruta_prueba):
    datos_serial = [] # Resetear lista
    procesar_imagen_multi_objetivo(ruta_prueba)
    df_resultado = pd.DataFrame(datos_serial)
    display(df_resultado)
else:
    print("[!] Asegúrate de tener 'prueba.jpg' en tus archivos.")

In [ ]:
# Descomprimir el archivo en una carpeta llamada 'fotos_serial'
!unzip -q fotosserial.zip -d fotos_serial

# Verificar cuántos archivos se extrajeron realmente
import os
print(f"¡Extracción completada! Total de archivos en la carpeta: {len(os.listdir('fotos_serial'))}")

In [ ]:
import glob

# 1. Definir la ruta donde están todas tus imágenes
# Si subiste un ZIP y lo descomprimiste en una carpeta llamada 'fotos_serial' usa esa ruta.
# Cambia 'fotos_serial' por el nombre real de tu carpeta.
carpeta_imagenes = 'fotos_serial/'

# Buscar extensiones comunes de imagen (.jpg, .jpeg, .png)
patrones = [os.path.join(carpeta_imagenes, '*.jpg'),
            os.path.join(carpeta_imagenes, '*.jpeg'),
            os.path.join(carpeta_imagenes, '*.png')]

lista_imagenes = []
for patron in patrones:
    lista_imagenes.extend(glob.glob(patron))

print(f"Se encontraron {len(lista_imagenes)} imágenes listas para procesamiento.")

# Resetear la base de datos para el procesamiento masivo
datos_serial = []

# 2. Correr el Pipeline en lote
print("Iniciando procesamiento masivo... Esto puede tomar un par de minutos.")
for idx, ruta_img in enumerate(lista_imagenes):
    try:
        procesar_imagen_multi_objetivo(ruta_img)
        if (idx + 1) % 10 == 0:
            print(f"Progreso: {idx + 1}/{len(lista_imagenes)} fotos completadas.")
    except Exception as e:
        print(f"[!] Error procesando la imagen {ruta_img}: {str(e)}")

# 3. Consolidar y Exportar a un archivo CSV
df_final = pd.DataFrame(datos_serial)

# Limpiar un poco el formato de los gadgets para el reporte ejecutivo
if not df_final.empty:
    df_final['Gadgets_Detectados'] = df_final['Gadgets_Detectados'].str.capitalize()

# Guardar en el almacenamiento local de Colab
ruta_csv = 'reporte_analitico_serial.csv'
df_final.to_csv(ruta_csv, index=False, encoding='utf-8-sig')

print("\n" + "="*40)
print("¡PROCESAMIENTO EN LOTE FINALIZADO CON ÉXITO!")
print(f"Dataset generado con {len(df_final)} filas de atletas analizados.")
print(f"El archivo '{ruta_csv}' está listo para descargarse en el panel izquierdo.")
print("="*40)

# Mostrar las primeras filas del resultado final
display(df_final.head(20))

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# 1. Cargar y estandarizar los datos extraídos
ruta_csv = 'reporte_analitico_serial.csv'
if not os.path.exists(ruta_csv):
    raise FileNotFoundError(f"No se encontró el archivo {ruta_csv}. Asegúrate de que el nombre sea correcto.")

df = pd.read_csv(ruta_csv)

# Limpieza rápida de strings para asegurar consistencia en los gráficos
df['Gadgets_Detectados'] = df['Gadgets_Detectados'].str.replace('A ', '').str.strip().str.capitalize()
df['Nivel_Emocion'] = df['Nivel_Emocion'].str.strip().str.capitalize()

# Configuración estética general (Estilo Clean Tech)
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'sans-serif'
palette_tech = ["#1a365d", "#2b6cb0", "#4299e1", "#ed8936", "#48bb78"]

# Crear carpeta para guardar las visualizaciones si no existe
os.makedirs('visualizations', exist_ok=True)

print("Generando gráficos ejecutivos de negocio...")

# ==========================================
# GRÁFICO 1: PENETRACIÓN DE MERCADO (GADGETS)
# ==========================================
plt.figure(figsize=(9, 5))
gadget_counts = df['Gadgets_Detectados'].value_counts()
ax1 = sns.barplot(x=gadget_counts.values, y=gadget_counts.index, palette=palette_tech, hue=gadget_counts.index, legend=False)

# Añadir etiquetas de porcentaje/conteo en las barras
total_atletas = len(df)
for i, v in enumerate(gadget_counts.values):
    porcentaje = (v / total_atletas) * 100
    ax1.text(v + (total_atletas*0.005), i, f"{v} ({porcentaje:.1f}%)", va='center', fontweight='bold', color='#2d3748')

plt.title('Market Share & Wearable Penetration among Athletes', fontsize=14, fontweight='bold', pad=15, color='#1a365d')
plt.xlabel('Number of Detections', fontsize=11, fontweight='bold', color='#4a5568')
plt.ylabel('Gadget Category', fontsize=11, fontweight='bold', color='#4a5568')
plt.tight_layout()
plt.savefig('visualizations/gadget_penetration.png', dpi=300)
plt.show()

# ==========================================
# GRÁFICO 2: INDEXACIÓN OPERATIVA (BIBS)
# ==========================================
plt.figure(figsize=(7, 5))
df['Bib_Status'] = df['Bib_Corredor'].apply(lambda x: 'Bib OCR Detected' if str(x).isdigit() else 'No Clear Bib Visible')
bib_counts = df['Bib_Status'].value_counts()

plt.pie(bib_counts.values, labels=bib_counts.index, autopct='%1.1f%%', startangle=140,
        colors=['#2b6cb0', '#e2e8f0'], textprops={'fontweight': 'bold', 'color': '#2d3748'},
        wedgeprops={'edgecolor': 'w', 'linewidth': 2})

plt.title('Operational Efficiency: Automated Bib Detection Rate', fontsize=14, fontweight='bold', pad=15, color='#1a365d')
plt.tight_layout()
plt.savefig('visualizations/bib_detection_rate.png', dpi=300)
plt.show()

# ==========================================
# GRÁFICO 3: DISTRIBUCIÓN DE EMOCIONES (MARKETING ROI)
# ==========================================
plt.figure(figsize=(8, 5))
emotion_counts = df['Nivel_Emocion'].value_counts()
ax3 = sns.countplot(data=df, x='Nivel_Emocion', order=emotion_counts.index, palette=["#48bb78", "#2b6cb0", "#e53e3e"])

# Añadir conteos arriba de las columnas
for p in ax3.patches:
    altura = p.get_height()
    ax3.annotate(f'{int(altura)}', (p.get_x() + p.get_width() / 2., altura),
                ha='center', va='center', xytext=(0, 8), textcoords='offset points', fontweight='bold', color='#2d3748')

plt.title('Audience Sentiment Analysis for Campaign Optimization', fontsize=14, fontweight='bold', pad=15, color='#1a365d')
plt.xlabel('Detected Emotional State', fontsize=11, fontweight='bold', color='#4a5568')
plt.ylabel('Frames Captured', fontsize=11, fontweight='bold', color='#4a5568')
plt.ylim(0, max(emotion_counts.values) * 1.15) # Dar espacio arriba para las etiquetas
plt.tight_layout()
plt.savefig('visualizations/sentiment_distribution.png', dpi=300)
plt.show()

print("\n" + "="*40)
print("¡VISUALIZACIONES GENERADAS Y GUARDADAS EN LA CARPETA 'visualizations/'!")
print("="*40)